## making the list of TSS regions
Causion!: The final output is saved as TF_GRN_matrix.txt on Zenodo. You can use this file to apply CATaN to your data.
- downloaded gtf file from https://www.gencodegenes.org/human/release_26lift37.html
- included lines marked as both "basic" and "transcript"

## Linux

TSS_list_tag_basic.bed: TSS list of genes tagged as "basic" of gencode (release 26 GRCh 37) 

In [ ]:
bedtools sort -i TSS_list_tag_basic.bed > TSS_list_sort_tag_basic.bed
TSS_region="~/TF_diagonalCCA/TSS_list_sort_tag_basic.bed"

bedtools slop -i ${TSS_region} -g ~/LDSC/chrom.sizes -l 1000000 -r 1000000 -s >TSS_list_exp_l1000000_r1000000_tag_basic.bed

In [ ]:
dd="path/to/TF_ChIP-seq_data"

peak_list=`less -S $dd/info/data_with_all_autosome.names`
TSS_region="~/TF_diagonalCCA/TSS_list_sort_tag_basic.bed"
cat $TSS_region |  sort -k4,4 - > ~/TF_diagonalCCA/TSS_list_sort_tag_basic_genenamesort.bed
gene_region="~/TF_diagonalCCA/TSS_list_exp_l1000000_r1000000_tag_basic.bed"
savepath="~/TF_diagonalCCA/TF_gene_distance_minus1000000_1000000"

mkdir $savepath
rm -f ${savepath}/TSS_peak_distance.bed

for peak_name in ${peak_list}
do
peak_file=${dd}/bed_hg19/${peak_name}.hg19.bed
bedtools intersect -wa -wb -a ${gene_region} -b ${peak_file} | sort -k4,4> ${savepath}/${peak_name}_genenamesort.bed
join -j 4 -t "$(printf '\011')" /home/imgtaka/TF_diagonalCCA/TSS_list_sort_tag_basic_genenamesort.bed ${savepath}/${peak_name}_genenamesort.bed | awk 'BEGIN{FS="\t";OFS="\t"}{print $2, $3, $4, $1, $5, $6, $12, $13, $14}' > ${savepath}/${peak_name}_TSS_peak.bed
cat ${savepath}/${peak_name}_TSS_peak.bed | awk -v peak_name=${peak_name} 'BEGIN{FS="\t";OFS="\t"}{if ($3 < $8) 
    print $0, $8-$3, peak_name;
else if($9 < $2) 
    print $0, $2-$9, peak_name; 
else 
    print $0, "1", peak_name}'>> ${savepath}/TSS_peak_distance.bed
done

In [ ]:
data_path="~/TF_diagonalCCA/TF_gene_distance_minus1000000_1000000/TSS_peak_distance.bed"
cat $data_path | awk 'BEGIN{FS="\t";OFS="\t"}{print $4"-"$11, $10}' | sort -t$'\t' -k1,1 -k 2n | bgzip -c > ~/TF_matrix/TF_matrix_tuning/TF_distance_minus1000000_1000000/TF_distance_minus1000000_1000000_sort.txt.gz

In [ ]:
zcat ~/TF_matrix/TF_matrix_tuning/TF_distance_minus1000000_1000000/TF_distance_minus1000000_1000000_sort.txt.gz | awk 'BEGIN{FS="\t";OFS="\t";OFMT="%.20f"}{x=$2; y=-x/10000; $3= 2^y;print $0}' | sort -t$'\t' -k1,1 -k 2n| bgzip -c > ~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_exp_d0_10K_sort_ver2.txt.gz
zcat ~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_exp_d0_10K_sort_ver2.txt.gz | awk 'BEGIN{FS="\t";OFS="\t";OFMT="%.20f"}{sum[$1]+=$3} END{for(i in sum) print i, sum[i]}'  | bgzip -c >~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_exp_d0_10K_sum_ver2.txt.gz
zcat ~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_exp_d0_10K_sum_ver2.txt.gz | awk -F'[-\t]' 'BEGIN{OFMT="%.20f"}{print $1, $2, $3}'  | bgzip -c > ~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_exp_d0_10K_sum_separate_ver2.txt.gz

## R

In [ ]:
library(dplyr)
library(tidyverse)
library(data.table)
library(ggplot2)



options(stringsAsFactors=F)
options(scipen=100)

In [ ]:
df=fread("~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_exp_d0_10K_sum_separate_ver2.txt.gz", stringsAsFactors=FALSE, data.table=FALSE, header = FALSE)
colnames(df)=c("gene_name", "TF_name", "distance")
df2=df %>% pivot_wider(names_from = "TF_name", values_from="distance", values_fill=0)
write.table(data.frame(gene=rownames(df2), df2), paste0("~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_flag_matrix.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

#sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.8 (Ootpa)

Matrix products: default
BLAS/LAPACK: /rshare1/ZETTAI_path_WA_slash_home_KARA/home/imgtaka/miniconda3/envs/renv/lib/libopenblasp-r0.3.26.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=ja_JP.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=ja_JP.UTF-8        LC_COLLATE=ja_JP.UTF-8    
 [5] LC_MONETARY=ja_JP.UTF-8    LC_MESSAGES=ja_JP.UTF-8   
 [7] LC_PAPER=ja_JP.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=ja_JP.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Tokyo
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] digest_0.6.35   IRdisplay_1.1   utf8_1.2.4      base64enc_0.1-3
 [5] fastmap_1.1.1   glue_1.7.0      htmltools_0.5.8 repr_1.1.7     
 [9] lifecycle_1.0.4 cli_3.6.2       fansi_1.0.6     vctrs_0.6.5    
[13] pbdZMQ_0.3-11   compiler_4.3.3  tools_4.3.3     evaluate_0.23  
[17] pillar_1.9.0    crayon_1.5.2    rlang_1.1.3     jsonlite_1.8.8 
[21] IRkernel_1.3.2  uuid_1.2-0     